# 🏥 Healthcare AI Model Training on Google Colab

This notebook trains a healthcare AI model using free T4 GPU on Google Colab.

## Instructions:
1. Upload your `medical_data_combined.jsonl` file
2. Run all cells
3. Download the trained model

**Training time:** ~10-15 minutes on T4 GPU

In [ ]:
# Step 1: Install dependencies
!pip install -q transformers datasets peft accelerate

In [ ]:
# Step 2: Upload your dataset
from google.colab import files
uploaded = files.upload()

# Verify the file was uploaded
import os
for filename in uploaded.keys():
    print(f'Uploaded: {filename}')

In [ ]:
# Step 3: Training script
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model
import torch
import json

print("🏥 Healthcare AI Training on Google Colab")
print("=" * 50)

# Configuration
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
MAX_SEQ_LENGTH = 512

# Check GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"📱 Device: {device}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")

# Load Model and Tokenizer
print("\n📥 Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    trust_remote_code=True,
).to(device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# Setup LoRA
print("🔧 Setting up LoRA...")
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
print(f"📊 Trainable parameters: {model.print_trainable_parameters()}")

# Healthcare Prompt Template
HEALTHCARE_PROMPT = """### Instruction:
You are a helpful healthcare assistant. Provide accurate, empathetic health information.
Always include a disclaimer that this is not medical advice.
Be concise and professional.

User Query: {}

### Response:
{}"""

# Load Dataset
print("\n📚 Loading medical dataset...")
texts = []
with open("medical_data_combined.jsonl", "r") as f:
    for line in f:
        data = json.loads(line.strip())
        text = HEALTHCARE_PROMPT.format(data['instruction'], data['output'])
        texts.append(text)

print(f"✅ Loaded {len(texts)} training examples")

# Create Dataset Class
class MedicalDataset:
    def __init__(self, texts, tokenizer, max_length):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        encoding = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt",
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels": encoding["input_ids"].squeeze(),
        }

dataset = MedicalDataset(texts, tokenizer, MAX_SEQ_LENGTH)

# Training Arguments
training_args = TrainingArguments(
    output_dir="healthcare_model",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=5,
    save_strategy="epoch",
    optim="adamw_torch",
    warmup_ratio=0.1,
    report_to="none",
)

# Data Collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

# Initialize Trainer
print("🚀 Starting training...")
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=data_collator,
)

# Train
trainer.train()

# Save Model
print("\n💾 Saving healthcare model...")
model.save_pretrained("healthcare_assistant_lora")
tokenizer.save_pretrained("healthcare_assistant_lora")

print("\n" + "=" * 50)
print("✅ Training complete!")
print("📁 Model saved to: healthcare_assistant_lora/")
print("=" * 50)

In [ ]:
# Step 4: Download the trained model
import shutil

# Zip the model
shutil.make_archive("healthcare_assistant_lora", "zip", "healthcare_assistant_lora")

# Download
files.download("healthcare_assistant_lora.zip")

print("📥 Model downloaded!")
print("\nNext steps:")
print("1. Extract the zip file")
print("2. Place the 'healthcare_assistant_lora' folder in your ai-model/ directory")
print("3. Run the inference server")